In [1]:
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [2]:
X_full = np.load('/lustre09/project/6081099/reem2005/DATASET/X_windows.npy')
y_full = np.load('/lustre09/project/6081099/reem2005/DATASET/y_windows.npy')
groups_full = np.load('/lustre09/project/6081099/reem2005/DATASET/groups.npy')

print("Before removing subject 108:")
print("X shape:", X_full.shape, "| Subjects:", np.unique(groups_full))

# Completely exclude subject 108 from the dataset (not just as a test fold)
keep_mask = groups_full != 108
X = X_full[keep_mask]
y = y_full[keep_mask]
groups = groups_full[keep_mask]

print("\nAfter removing subject 108:")
print("X shape:", X.shape, "| Subjects:", np.unique(groups))

Before removing subject 108:
X shape: (28727, 100, 18) | Subjects: [101 102 103 104 105 106 107 108]

After removing subject 108:
X shape: (25094, 100, 18) | Subjects: [101 102 103 104 105 106 107]


In [3]:
BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7
GRAD_CLIP_NORM = 5.0

INPUT_WINDOW = X.shape[1]
INPUT_FEATURES = X.shape[2]
NUM_CLASSES = len(np.unique(y))
RANDOM_SEED = 42

print(f"Window: {INPUT_WINDOW} | Features: {INPUT_FEATURES} | Classes: {NUM_CLASSES} | Subjects: {len(np.unique(groups))}")

Window: 100 | Features: 18 | Classes: 8 | Subjects: 7


In [4]:
class HARDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [9]:
class DNNClassifier(nn.Module):
    def __init__(self, input_window, input_features, num_classes):
        super().__init__()
        input_dim = input_window * input_features
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)

In [5]:
class CNN1DClassifier(nn.Module):
    def __init__(self, input_window, input_features, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_features, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.3),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, input_features, input_window)
            flat_dim = self.conv(dummy).flatten(1).shape[1]
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(flat_dim, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        return self.classifier(x)

In [6]:
class RNNClassifier(nn.Module):
    def __init__(self, input_window, input_features, num_classes, hidden_size=128):
        super().__init__()
        self.rnn = nn.RNN(input_features, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.dropout(out[:, -1, :])
        return self.classifier(out)

In [7]:
class DeepConvLSTMClassifier(nn.Module):
    def __init__(self, input_window, input_features, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_features, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
        )
        self.lstm = nn.LSTM(64, 128, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(128, 32)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        out = self.dropout2(self.relu(self.fc1(out)))
        return self.classifier(out)

In [11]:
import sys
sys.path.insert(0, "/lustre09/project/6081099/reem2005/ISWC22-HAR-main")
from models.TinyHAR import TinyHAR_Model

class TinyHARClassifier(nn.Module):
    def __init__(self, input_window, input_features, num_classes, filter_num=32):
        super().__init__()
        input_shape = (1, 1, input_window, input_features)
        self.model = TinyHAR_Model(
            input_shape=input_shape, number_class=num_classes, filter_num=filter_num,
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        return self.model(x)

In [12]:
MODEL_REGISTRY = {
    "dnn": DNNClassifier,
    "cnn1d": CNN1DClassifier,
    "rnn": RNNClassifier,
    "deepconv_lstm": DeepConvLSTMClassifier,
    "tinyhar": TinyHARClassifier,
}
print("Models to compare:", list(MODEL_REGISTRY.keys()))

Models to compare: ['dnn', 'cnn1d', 'rnn', 'deepconv_lstm', 'tinyhar']


In [13]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, f1_macro, all_labels, all_preds

In [14]:
def run_loso_experiment(model_name, epochs=EPOCHS, verbose=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    unique_subjects = np.unique(groups)

    fold_f1s, fold_accuracies = [], []
    all_y_true, all_y_pred = [], []

    for fold, test_subject in enumerate(unique_subjects, 1):
        train_subjects_all = unique_subjects[unique_subjects != test_subject]

        rng = np.random.RandomState(RANDOM_SEED + fold)
        val_subject = rng.choice(train_subjects_all)
        train_subjects = train_subjects_all[train_subjects_all != val_subject]

        train_mask = np.isin(groups, train_subjects)
        val_mask = groups == val_subject
        test_mask = groups == test_subject

        X_train, y_train = X[train_mask], y[train_mask]
        X_val, y_val = X[val_mask], y[val_mask]
        X_test, y_test = X[test_mask], y[test_mask]

        train_loader = DataLoader(HARDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(HARDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(HARDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

        model = MODEL_REGISTRY[model_name](INPUT_WINDOW, INPUT_FEATURES, NUM_CLASSES).to(device)

        class_counts = np.bincount(y_train, minlength=NUM_CLASSES)
        class_weights = torch.tensor(
            len(y_train) / (NUM_CLASSES * np.maximum(class_counts, 1)), dtype=torch.float32
        ).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)

        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

        best_val_f1 = -1
        best_model_state = None
        patience_counter = 0

        for epoch in range(epochs):
            train_one_epoch(model, train_loader, optimizer, criterion, device)
            _, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
            scheduler.step(val_f1)

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    break

        model.load_state_dict(best_model_state)
        _, test_f1, y_true, y_pred = evaluate(model, test_loader, criterion, device)
        test_acc = accuracy_score(y_true, y_pred)

        fold_f1s.append(test_f1)
        fold_accuracies.append(test_acc)
        all_y_true.extend(y_true)
        all_y_pred.extend(y_pred)

        if verbose:
            print(f"  [{model_name}] Fold {fold} | test=subject{test_subject} | "
                  f"Acc={test_acc:.4f} | F1={test_f1:.4f}")

    return {
        "model_name": model_name,
        "fold_f1s": fold_f1s,
        "fold_accuracies": fold_accuracies,
        "mean_f1": np.mean(fold_f1s),
        "std_f1": np.std(fold_f1s),
        "mean_accuracy": np.mean(fold_accuracies),
        "std_accuracy": np.std(fold_accuracies),
        "y_true": all_y_true,
        "y_pred": all_y_pred,
    }

In [15]:
from pathlib import Path
import gc

RESULTS_OUTPUT = Path("/lustre09/project/6081099/reem2005/DATASET/pamap2_no108_results.pkl")

if RESULTS_OUTPUT.exists():
    all_results = pd.read_pickle(RESULTS_OUTPUT)
    print("Resuming from saved results:", list(all_results.keys()))
else:
    all_results = {}

for model_name in MODEL_REGISTRY:
    if model_name in all_results:
        print(f"Skipping {model_name} — already done")
        continue

    print(f"\n{'='*60}\nTraining: {model_name}\n{'='*60}")
    t0 = time.time()
    result = run_loso_experiment(model_name, epochs=EPOCHS, verbose=True)
    elapsed = time.time() - t0
    result["elapsed_minutes"] = elapsed / 60
    all_results[model_name] = result

    print(f"Mean Acc: {result['mean_accuracy']:.4f} | Mean F1: {result['mean_f1']:.4f} ± {result['std_f1']:.4f} | Time: {elapsed/60:.1f} min")

    pd.to_pickle(all_results, RESULTS_OUTPUT)
    torch.cuda.empty_cache()
    gc.collect()


Training: dnn
  [dnn] Fold 1 | test=subject101 | Acc=0.6828 | F1=0.6856
  [dnn] Fold 2 | test=subject102 | Acc=0.6425 | F1=0.5886
  [dnn] Fold 3 | test=subject103 | Acc=0.7625 | F1=0.7577
  [dnn] Fold 4 | test=subject104 | Acc=0.7217 | F1=0.7004
  [dnn] Fold 5 | test=subject105 | Acc=0.8280 | F1=0.8213
  [dnn] Fold 6 | test=subject106 | Acc=0.7934 | F1=0.7977
  [dnn] Fold 7 | test=subject107 | Acc=0.8463 | F1=0.8113
Mean Acc: 0.7539 | Mean F1: 0.7375 ± 0.0782 | Time: 0.7 min

Training: cnn1d
  [cnn1d] Fold 1 | test=subject101 | Acc=0.7074 | F1=0.6773
  [cnn1d] Fold 2 | test=subject102 | Acc=0.6844 | F1=0.6365
  [cnn1d] Fold 3 | test=subject103 | Acc=0.7781 | F1=0.7718
  [cnn1d] Fold 4 | test=subject104 | Acc=0.7256 | F1=0.7251
  [cnn1d] Fold 5 | test=subject105 | Acc=0.7919 | F1=0.7858
  [cnn1d] Fold 6 | test=subject106 | Acc=0.7936 | F1=0.7931
  [cnn1d] Fold 7 | test=subject107 | Acc=0.8728 | F1=0.8657
Mean Acc: 0.7648 | Mean F1: 0.7508 ± 0.0715 | Time: 0.6 min

Training: rnn
  [rnn]

In [16]:
summary_df = pd.DataFrame([
    {
        "model": name,
        "mean_accuracy": res["mean_accuracy"],
        "mean_f1_macro": res["mean_f1"],
        "std_f1_macro": res["std_f1"],
        "time_minutes": res["elapsed_minutes"],
    }
    for name, res in all_results.items()
]).sort_values("mean_f1_macro", ascending=False).reset_index(drop=True)

print("PAMAP2 Model Comparison — WITHOUT Subject 108 (7 subjects, 7-fold LOSO)")
summary_df

PAMAP2 Model Comparison — WITHOUT Subject 108 (7 subjects, 7-fold LOSO)


,model,mean_accuracy,mean_f1_macro,std_f1_macro,time_minutes
0,cnn1d,0.764839,0.750760,0.071480,0.601974
1,dnn,0.753901,0.737508,0.078157,0.692841
2,deepconv_lstm,0.749020,0.718501,0.132149,0.941457
3,tinyhar,0.719734,0.716649,0.050686,5.993181
4,rnn,0.668225,0.637009,0.093466,0.762816
